# Corrected retrain — the list San Diego's screen ignores

Rebuilds the candidate set as **only intersections below the City's 5-crash screen**
(`crashes_feat < 5`), then refits the frozen XGBoost Tweedie model on two feature sets —
**crash-only (A)** and **crash + infrastructure (D)** — with genuine 5-fold out-of-fold
scoring, and compares them to the persistence baseline and to the City ($0 here, since the
City never looks at these sites).

**Runtime → Change runtime type → GPU is optional** (XGBoost is fast on CPU too).

### Upload these 6 files (drag them into the Files panel, or use the upload cell below)
| file | on your machine, in |
|---|---|
| `candidate_panel.parquet` | `data\\model\\verified_run\\` |
| `feature_table.parquet` | `data\\model\\verified_run\\` |
| `oof_scores.parquet` | `data\\model\\verified_run\\` |
| `infra_features.parquet` | `data\\model\\` |
| `frozen_params.json` | `data\\model\\` |
| `crashes_4326.parquet` | `data\\proc\\` |

Run the cells top to bottom.


In [ ]:
# 1. deps (xgboost is usually preinstalled on Colab; this makes sure)
!pip -q install xgboost 2>/dev/null
import xgboost, sklearn, scipy, pandas, numpy
print("xgboost", xgboost.__version__, "| ready")


In [ ]:
# 2. upload the 6 files (pick all 6 at once). They land flat in the working dir.
from google.colab import files
up = files.upload()
print("\nuploaded:", sorted(up))
need = {"candidate_panel.parquet","feature_table.parquet","oof_scores.parquet",
        "infra_features.parquet","frozen_params.json","crashes_4326.parquet"}
missing = need - set(up)
print("MISSING:" , missing if missing else "none — good to go")


In [ ]:
# 3. corrected retrain + comparison
import json, numpy as np, pandas as pd, xgboost as xgb
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.model_selection import GroupKFold, StratifiedKFold

# --- repo methodology constants ---
COST = 5_175_524      # $ per KSI event (FHWA-SA-25-021, 24.3% fatal share)
EFF  = 0.30           # treatment effectiveness
PROG = 3_200_000      # top-500 program cost after 90% HSIP
SEED = 42; NF = 5; BUF = 76.2; CITY_MIN = 5
FS, FE = pd.Timestamp("2016-01-01"), pd.Timestamp("2021-12-31")   # verified feature window

CRASH = ["crashes_36mo","crashes_72mo","ped_crashes_72mo","bike_crashes_72mo","broadside_72mo",
 "left_turn_72mo","dui_72mo","night_72mo","ped_row_violation_72mo","years_since_last_crash",
 "distinct_crash_days_72mo","worst_severity_72mo","crash_trend_slope","emergence_velocity",
 "emergence_acceleration","mann_kendall_tau","changepoint_prob","ewma_crashes","momentum_ratio","covid_period_share"]

# --- load ---
panel = pd.read_parquet("candidate_panel.parquet", columns=["intersection_id","lon","lat","KSI_label"])
feats = pd.read_parquet("feature_table.parquet")
infra = pd.read_parquet("infra_features.parquet")
oof   = pd.read_parquet("oof_scores.parquet", columns=["intersection_id","spatial_block","persistence_baseline_score"])
fp = json.load(open("frozen_params.json"))["frozen"]
params = dict(objective="reg:tweedie",
    tweedie_variance_power=float(fp["tweedie_variance_power"]), max_depth=int(fp["max_depth"]),
    learning_rate=float(fp["learning_rate"]), n_estimators=int(fp["n_estimators"]),
    reg_alpha=float(fp["reg_alpha"]), reg_lambda=float(fp["reg_lambda"]),
    min_child_weight=int(fp["min_child_weight"]), subsample=float(fp["subsample"]),
    colsample_bytree=float(fp["colsample_bytree"]), random_state=42, verbosity=0)

df = (panel.merge(feats, on="intersection_id", how="left")
           .merge(infra, on="intersection_id", how="left")
           .merge(oof,   on="intersection_id", how="left"))

# --- crashes_feat over the verified window, via lon/lat snap (76.2 m) ---
cr = pd.read_parquet("crashes_4326.parquet", columns=["date","STATE_HWY_IND","lon","lat"])
cr["date"] = pd.to_datetime(cr["date"])
cr = cr[cr["STATE_HWY_IND"].astype(str).str.upper() != "Y"]
cr = cr[(cr.date >= FS) & (cr.date <= FE)]
lat0 = np.deg2rad(df.lat.mean()); R = 6371000.0
xy = lambda d: np.column_stack([R*np.cos(lat0)*np.deg2rad(d.lon.values), R*np.deg2rad(d.lat.values)])
dd, ii = cKDTree(xy(df)).query(xy(cr), k=1, workers=-1); w = dd <= BUF
df["crashes_feat"] = df.intersection_id.map(
    pd.Series(df.intersection_id.values[ii[w]]).value_counts()).fillna(0).astype(int)

# --- CORRECTED candidate set: below the City screen ---
corr = df[df.crashes_feat < CITY_MIN].reset_index(drop=True)
print(f"corrected candidates (<{CITY_MIN} crashes): {len(corr)}  "
      f"| pos>=1={int((corr.KSI_label>=1).sum())}  pos>=2={int((corr.KSI_label>=2).sum())}\n")

infra_cols = [c for c in infra.columns if c != "intersection_id" and corr[c].fillna(0).var() > 0]
SETS = {"A_crash_only": CRASH, "D_crash_plus_infra": CRASH + infra_cols}

y = corr.KSI_label.values.astype(float)
groups = corr.spatial_block.values
base = corr.persistence_baseline_score.values

def folds(mode):
    if mode == "random":
        return list(StratifiedKFold(NF, shuffle=True, random_state=SEED).split(y, (y>=2).astype(int)))
    return list(GroupKFold(NF).split(y, y, groups=groups))

def oof_xgb(X, mode):
    o = np.full(len(y), np.nan)
    for tr, te in folds(mode):
        m = xgb.XGBRegressor(**params); m.fit(X[tr], y[tr]); o[te] = m.predict(X[te])
    return o

def money(scores, T, K=500):
    kl = y[np.argsort(-scores)[:K]]; ev = int(kl[kl>=T].sum())
    return ev, ev*COST*EFF

def spear(s): return float(stats.spearmanr(s, y).correlation)

print("================ CORRECTED SET — model vs baseline (city = $0 here) ================")
print("Every dollar below is EXTRA vs the City: these are sites its 5-crash screen ignores.\n")
results = {}
for sn, cols in SETS.items():
    X = corr[cols].fillna(0.0).values.astype(float)
    for mode in ["random", "spatial"]:
        mo = oof_xgb(X, mode)
        line = f"{sn:20s} {mode:7s} rho={spear(mo):+.4f} "
        row = {"spearman": round(spear(mo),4)}
        for T in (1,2):
            me, md = money(mo, T); be, bd = money(base, T)
            line += f"| >={T}KSI model ${md/1e6:5.1f}M({me}ev) base ${bd/1e6:5.1f}M({be}ev)  vs-base {(md-bd)/1e6:+.1f}M "
            row |= {f"ge{T}_model_$M": round(md/1e6,1), f"ge{T}_base_$M": round(bd/1e6,1),
                    f"ge{T}_vs_baseline_$M": round((md-bd)/1e6,1)}
        results[f"{sn}__{mode}"] = row
        print(line)
print(f"\n{'persistence_baseline':20s} {'ref':7s} rho={spear(base):+.4f} "
      f"| >=1 ${money(base,1)[1]/1e6:.1f}M | >=2 ${money(base,2)[1]/1e6:.1f}M")

json.dump(results, open("corrected_retrain_results.json","w"), indent=2)

# --- deliverable: the list the City ignores (set-D model, spatial OOF) ---
X = corr[SETS["D_crash_plus_infra"]].fillna(0.0).values.astype(float)
corr["model_score"] = oof_xgb(X, "spatial")
(corr.sort_values("model_score", ascending=False).head(500)
     [["intersection_id","lon","lat","crashes_feat","KSI_label","model_score"]]
     .to_csv("city_ignored_top500.csv", index=False))
print("\nSaved: city_ignored_top500.csv  (500 sites the City's >=5 screen never flags)")
print("Saved: corrected_retrain_results.json")
try:
    files.download("city_ignored_top500.csv")
except Exception:
    pass


## How to read the output

- **`>=1KSI model $ ... vs-base +X.XM`** — the extra prevented harm (over the 3-yr window)
  from the model's top-500 on the city-ignored set. Positive `vs-base` = the model beats the
  persistence heuristic. Compare **A_crash_only** vs **D_crash_plus_infra**: if D's dollars /
  rho are consistently higher on **both** `random` and `spatial`, infrastructure features are
  the lift you were looking for.
- **Every model dollar is "extra vs the City"** — the City catches $0 here by construction.
- **`city_ignored_top500.csv`** downloads automatically — that's the deliverable: 500
  intersections San Diego's 5-crash screen will never flag, ranked by predicted KSI risk.

If `D` doesn't clearly beat `A`, the honest read is that crash-history + current infra still
isn't enough signal, and the next lever is **exposure/ADT data** — not a bigger model.
